In [38]:
import pyspark 
from pyspark.sql import SparkSession 
spark = SparkSession\
    .builder.master("local[*]")\
    .appName('test')\
    .getOrCreate()

In [39]:
df = spark.read.option("heder", "true").parquet("data/raw/yellow_tripdata_2025-11.parquet")

In [40]:
df.head(5)

[Row(VendorID=7, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), passenger_count=1, trip_distance=1.68, RatecodeID=1, store_and_fwd_flag='N', PULocationID=43, DOLocationID=186, payment_type=1, fare_amount=14.9, extra=0.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.15, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 49, 7), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 1, 1, 22), passenger_count=1, trip_distance=2.28, RatecodeID=1, store_and_fwd_flag='N', PULocationID=142, DOLocationID=237, payment_type=1, fare_amount=14.2, extra=1.0, mta_tax=0.5, tip_amount=4.99, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=24.94, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2025, 11, 1,

In [41]:
input_path = "data/raw/yellow_tripdata_2025-11.parquet"

df = spark.read.parquet(input_path)

df = df.repartition(4)

df.write.parquet("data/pd/yellow/yellow_tripdata_2025-11")

[Stage 67:======================================>                  (8 + 4) / 12]

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/home/sangam/spark/spark_project/data/pd/yellow/yellow_tripdata_2025-11 already exists. Set mode as "overwrite" to overwrite the existing path.

In [ ]:
df.printSchema()

In [ ]:
df.createOrReplaceTempView('trips_data')

In [ ]:
# How many taxi trips were there on the 15th of November?
spark.sql("""
SELECT
    COUNT(*) AS total_trips
FROM trips_data
WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

In [ ]:
#What is the length of the longest trip in the dataset in hours?

spark.sql("""
SELECT
    MAX(
        (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600
    ) AS longest_trip_hours
FROM trips_data
""").show()

In [ ]:
! wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv


In [42]:
df_zone = spark.read.csv("data/temp/taxi_zone_lookup.csv", header=True)


In [45]:
df_zone.write.mode("overwrite").parquet("data/temp/taxi_zone_lookup.parquet")

In [46]:
df_zone.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [47]:
df.show()

[Stage 72:==========================================>              (9 + 3) / 12]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [57]:
spark.sql("""
WITH zone_counts AS (
    SELECT
        pu.Zone AS pickup_zone,
        COUNT(*) AS trip_count
    FROM trips_data t
    LEFT JOIN zones pu
        ON t.PULocationID = pu.LocationID
    GROUP BY pu.Zone
)
SELECT *
FROM zone_counts
WHERE trip_count = (SELECT MIN(trip_count) FROM zone_counts)
""").show(truncate=False)

[Stage 90:>                                                         (0 + 4) / 4]

+---------------------------------------------+----------+
|pickup_zone                                  |trip_count|
+---------------------------------------------+----------+
|Eltingville/Annadale/Prince's Bay            |1         |
|Governor's Island/Ellis Island/Liberty Island|1         |
|Arden Heights                                |1         |
+---------------------------------------------+----------+

